# Cohort: cardiac surgery x Fitbit

Defines the analysis cohort — participants with EHR data **and** Fitbit data who have a procedure
that is a descendant of OMOP concept `4301351`. Generated from the Verily Workbench cohort
builder (cohort `CardiacSurgxFitbit`, dataset `AllOfUsControlledTierDatasetv9`,
built `20260909_082818`), then adapted to this repo's conventions.

**Governance.** Outputs are stripped on commit, so anything rendered here stays local. Even so,
prefer aggregates over record listings — see the `allofus-data-governance` skill. Figures belong
in `cardiac_surg_fitbit_figures/`; query results are cached in `_cache/`. Both are git-ignored.


In [ ]:
import sys
from pathlib import Path

import pandas

# Locate the repo root -- the directory holding srp_common.py -- by walking up from
# cwd, so this works whether the kernel starts in the task directory or the repo root.
_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common

# Resolves the CDR from the environment, discovers the newest cohort index dataset,
# and creates this notebook's git-ignored figure and cache directories.
cfg = srp_common.init("cardiac_surg_fitbit")
CDR, CDR_INDEX = cfg.CDR, cfg.CDR_INDEX
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq


In [ ]:
person_sql = f"""

    SELECT
        st.birth_datetime,
        st.gender_concept_id,
        dt0.concept_name AS T_DISP_gender,
        st.person_id,
        st.sex_at_birth_concept_id,
        dt1.concept_name AS T_DISP_sex_at_birth 
    FROM
        `{CDR}`.person AS st 
    LEFT JOIN
        `{CDR}`.concept AS dt0 
            ON dt0.concept_id = st.gender_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt1 
            ON dt1.concept_id = st.sex_at_birth_concept_id 
    WHERE
        st.person_id IN (SELECT
            id 
        FROM
            `{CDR_INDEX}`.T_ENT_person 
        WHERE
            (has_ehr_data = true) 
            AND (has_fitbit = true) 
            AND (id IN (SELECT
                person_id AS primary_id 
            FROM
                `{CDR_INDEX}`.T_ENT_procedureOccurrence 
            WHERE
                procedure_concept_id IN (SELECT
                    descendant 
                FROM
                    `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                WHERE
                    ancestor = 4301351 
                UNION
                ALL SELECT
                    4301351))))
"""

person_df = cached_gbq(person_sql, "person")


In [ ]:
procedure_occurrence_sql = f"""

    SELECT
        st.modifier_source_value,
        st.person_id,
        st.procedure_concept_id,
        st.procedure_datetime,
        st.procedure_type_concept_id,
        dt0.concept_name AS T_DISP_procedure_type_concept_name,
        st.procedure_source_concept_id,
        dt1.vocabulary_id AS T_DISP_source_vocabulary,
        dt2.concept_code AS T_DISP_standard_concept_code,
        dt2.concept_name AS T_DISP_standard_concept_name,
        dt2.vocabulary_id AS T_DISP_standard_vocabulary,
        st.visit_occurrence_id 
    FROM
        `{CDR}`.procedure_occurrence AS st 
    LEFT JOIN
        `{CDR}`.concept AS dt0 
            ON dt0.concept_id = st.procedure_type_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt1 
            ON dt1.concept_id = st.procedure_source_concept_id 
    LEFT JOIN
        `{CDR}`.concept AS dt2 
            ON dt2.concept_id = st.procedure_concept_id 
    WHERE
        st.procedure_occurrence_id IN (SELECT
            id 
        FROM
            `{CDR_INDEX}`.T_ESA_procedureOccurrence_person_id 
        WHERE
            (person_id IN (SELECT
                id 
            FROM
                `{CDR_INDEX}`.T_ENT_person 
            WHERE
                (has_ehr_data = true) 
                AND (has_fitbit = true) 
                AND (id IN (SELECT
                    person_id AS primary_id 
                FROM
                    `{CDR_INDEX}`.T_ENT_procedureOccurrence 
                WHERE
                    procedure_concept_id IN (SELECT
                        descendant 
                    FROM
                        `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                    WHERE
                        ancestor = 4301351 
                    UNION
                    ALL SELECT
                        4301351))))) 
                    AND (procedure_concept_id IN (SELECT
                        descendant 
                    FROM
                        `{CDR_INDEX}`.T_HAD_procedureConcept_default 
                    WHERE
                        ancestor = 4301351 
                    UNION
                    ALL SELECT
                        4301351)))
"""

procedure_occurrence_df = cached_gbq(procedure_occurrence_sql, "procedure_occurrence")


In [ ]:
# Aggregate summary only — no participant-level record listing.
# The original notebook ended each query with .head(5); eyeballing rows interactively is
# fine, but a listing that lands in saved output is a DUA problem, so the default here is
# shape + schema. Suppress small cells before reporting any count table externally.

for label, df in [("person", person_df), ("procedure_occurrence", procedure_occurrence_df)]:
    print(f"{label:22s} rows={len(df):>9,}  cols={df.shape[1]:>3}  "
          f"unique person_id={df['person_id'].nunique():>8,}")

print()
print("person columns             :", list(person_df.columns))
print("procedure_occurrence columns:", list(procedure_occurrence_df.columns))
